# Talmadge Insights & Relationship Analysis

This notebook analyzes the communication history stored in BigQuery to extract deep insights, optimal communication times, sentiment patterns, and topic clusters.

In [ ]:
import bigframes
import bigframes.pandas as bpd
import bigframes.ml.cluster as cluster
%load_ext bigframes

## Data Loading
We load the messages ingested by Dataflow into BigFrames for analysis.

In [ ]:
%%bqsql df_messages
SELECT 
    message_id,
    timestamp,
    sender,
    text,
    message_length
FROM `jsco-core-storage.talmadge_analytics.communication_history`
ORDER BY timestamp DESC
LIMIT 1000

In [ ]:
df_messages.head()

## Sentiment Analysis
Using BigQuery ML to predict sentiment for each message to map his emotional warmth over time.

In [ ]:
%%bqsql df_sentiment
SELECT
  timestamp,
  sender,
  text,
  ml_generate_text_llm_result AS sentiment
FROM
  ML.GENERATE_TEXT(
    MODEL `jsco-core-storage.talmadge_analytics.gemini_model`,
    (
      SELECT timestamp, sender, text, CONCAT('Analyze the sentiment of this message and return Positive, Neutral, or Negative: ', text) AS prompt
      FROM `jsco-core-storage.talmadge_analytics.communication_history`
      WHERE sender = 'Tad'
    ),
    STRUCT(
      0.2 AS temperature,
      10 AS max_output_tokens
    )
  )

## Clustering Conversational Topics
We use K-Means clustering on the message length and time of day to group message types.

In [ ]:
# Extract hour of day for clustering
df_clustering = df_messages[['message_length']]
df_clustering['hour'] = df_messages['timestamp'].dt.hour

kmeans = cluster.KMeans(n_clusters=3)
kmeans.fit(df_clustering)

# Predict clusters
predictions = kmeans.predict(df_clustering)
predictions.head()

## Conclusion

### Data Analysis Key Findings
- (Will populate after data is ingested and notebook is executed)

### Insights or Next Steps
- (Will populate after data is ingested and notebook is executed)